In [7]:
import os
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"
from unsloth import FastLanguageModel
import torch
from fastapi import FastAPI
import uvicorn

app = FastAPI()

# --- 1. 初始化模型（直接加载适配器，不合并） ---
model_path = "outputs_yue_qwen/checkpoint-10000"

print("🚀 正在加载粤语翻译模型 (4-bit + LoRA)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path, # 这样加载会自动处理基座和适配器
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# --- 2. 翻译逻辑 ---
def translate_yue_to_zh(text: str):
    prompt = f"<|im_start|>system\n你是一个地道的粤语翻译助手。<|im_end|>\n<|im_start|>user\n{text}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 128, 
        use_cache = True,
        repetition_penalty = 1.2 # 防止之前提到的循环问题
    )
    
    result = tokenizer.decode(outputs[0], skip_special_tokens = True)
    return result.split("assistant")[-1].strip()


🚀 正在加载粤语翻译模型 (4-bit + LoRA)...
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.558 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 339/339 [00:01<00:00, 316.95it/s]


unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [9]:
if __name__ == "__main__":
    res = translate_yue_to_zh('7月18日，在广州体育馆举办《着迷·陈洁仪》巡回演唱会广州站')
    print(res)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


7月18號，喺廣州體育館舉辦《著迷陳潔儀》巡迴演唱會廣州站


In [5]:
import os
# 跳过环境版本检查
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"

import torch
from unsloth import FastLanguageModel
from safetensors.torch import save_file # 直接使用底层存储工具

# 1. 加载模型（从你找到的根目录路径加载）
print("🚀 正在加载 LoRA 模型...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/workspace/outputs_yue_qwen/checkpoint-10000",
    max_seq_length = 2048,
    load_in_4bit = False, # 确保 16bit 加载，避免量化合并冲突
)

# 2. 合并
print("🛠️ 正在执行 merge_and_unload...")
model = model.merge_and_unload()

# 3. 暴力存盘：手动写入权重和配置
save_path = "/workspace/model_to_convert"
os.makedirs(save_path, exist_ok=True)

print(f"💾 正在绕过库逻辑，直接写入物理文件到 {save_path}...")
try:
    # A. 提取权重字典并直接保存为 safetensors 格式
    state_dict = model.state_dict()
    save_file(state_dict, os.path.join(save_path, "model.safetensors"), metadata={"format": "pt"})
    
    # B. 单独保存配置文件（这些是文本，不会触发权重转换报错）
    model.config.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    
    # C. 强制刷新磁盘缓存
    os.sync()
    print("✅ 物理存盘成功！现在可以去终端执行转换了。")
except Exception as e:
    print(f"❌ 存盘依然失败: {e}")

🚀 正在加载 LoRA 模型...
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.558 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 339/339 [00:00<00:00, 2758.24it/s]


./base_model does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
🛠️ 正在执行 merge_and_unload...
💾 正在绕过库逻辑，直接写入物理文件到 /workspace/model_to_convert...
✅ 物理存盘成功！现在可以去终端执行转换了。


In [6]:
import json
import os

config_path = "/workspace/model_to_convert/config.json"

if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = json.load(f)

    # 核心操作：删除量化配置段落
    if "quantization_config" in config:
        print(f"🗑️ 正在删除残留的量化配置: {config['quantization_config'].get('quant_method')}")
        del config["quantization_config"]
    
    # 确保模型类型和权重格式正确
    config["torch_dtype"] = "bfloat16"

    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    
    print("✅ config.json 修复完成！现在它是纯正的 BF16 配置了。")
else:
    print("❌ 找不到 config.json，请确认路径是否正确。")

🗑️ 正在删除残留的量化配置: bitsandbytes
✅ config.json 修复完成！现在它是纯正的 BF16 配置了。


In [ ]:
import os
import torch
from safetensors.torch import save_file, load_file

# 1. 路径设置
raw_model_path = "/workspace/model_to_convert"
clean_model_path = "/workspace/model_final_clean_v2"
os.makedirs(clean_model_path, exist_ok=True)

print("🧹 正在进行深度清洗，剔除所有量化元数据...")

# 2. 读取权重
input_file = os.path.join(raw_model_path, "model.safetensors")
state_dict = load_file(input_file)

# 3. 深度过滤
clean_state_dict = {}
for k, v in state_dict.items():
    # 只要包含这些关键字的键，全部扔掉
    if any(ignore in k for ignore in [".absmax", ".quant_state", ".nested", ".quant_map"]):
        continue
    
    # 核心：如果权重还是 uint8，说明合并失败了，这种权重 llama.cpp 无法处理
    if v.dtype == torch.uint8:
        print(f"⚠️ 警告: 发现未解压的权重 {k}，尝试转换格式...")
        # 这种转换通常不准，但在没有办法的情况下可以尝试
        v = v.to(torch.bfloat16) 
    
    clean_state_dict[k] = v

# 4. 保存
save_file(clean_state_dict, os.path.join(clean_model_path, "model.safetensors"), metadata={"format": "pt"})

# 5. 复制配置
import shutil
for file in ["config.json", "tokenizer.json", "tokenizer_config.json"]:
    src = os.path.join(raw_model_path, file)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(clean_model_path, file))

print(f"✅ 深度清洗完成！保留键值对: {len(clean_state_dict)}")

In [ ]:
# 确保你的模型变量名是 model，分词器是 tokenizer
model.save_pretrained("/workspace/yue_lora_only")
tokenizer.save_pretrained("/workspace/yue_lora_only")
print("✅ LoRA 权重已成功保存到 /workspace/yue_lora_only")